# Real-time earthquake analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from cassandrautils import getEarthquakeDF

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)
pd.set_option("display.max_columns", 30)

## 1. Load and overview

In [ ]:
raw = getEarthquakeDF()
assert raw is not None and not raw.empty, "earthquakedata is empty - is the earthquake producer + sink running?"

raw["event_time"] = pd.to_datetime(raw["event_time"])
raw["region"] = raw["place"].fillna("Unknown").str.split(", ").str[-1].str.strip()

print("Events collected: {:,}".format(len(raw)))
print("Time range (UTC): {} -> {}".format(raw["event_time"].min(), raw["event_time"].max()))
print("Magnitude range : {:.1f} -> {:.1f}".format(raw["magnitude"].min(), raw["magnitude"].max()))
raw["event_type"].value_counts().rename("events").to_frame()

The feed also includes non-earthquake events (quarry blasts, explosions, ice quakes). We keep only `earthquake` for the rest of the analysis.

In [ ]:
eq = raw[raw["event_type"] == "earthquake"].dropna(subset=["magnitude"]).copy()
eq["depth_class"] = pd.cut(eq["depth_km"], bins=[-10, 70, 300, 800],
                           labels=["Shallow (<70 km)", "Intermediate (70-300 km)", "Deep (>300 km)"])
eq[["place", "magnitude", "depth_km", "event_time", "alert_level", "tsunami"]].sort_values("event_time", ascending=False).head(10)

## 2. Where do earthquakes happen?

Each dot is an earthquake. Size shows magnitude and color shows depth. Without drawing any map, the tectonic plate boundaries
appear on their own, especially the **Pacific Ring of Fire** around Japan, Indonesia and the Americas.

In [ ]:
CITIES = {"Tokyo": (35.6762, 139.6503), "Ho Chi Minh City": (10.8231, 106.6297)}

fig, ax = plt.subplots(figsize=(16, 8))
sc = ax.scatter(eq["longitude"], eq["latitude"], s=(eq["magnitude"].clip(lower=0) ** 2.5) * 2,
                c=eq["depth_km"], cmap="viridis_r", alpha=0.6, edgecolor="k", linewidth=0.2)
for name, (lat, lon) in CITIES.items():
    ax.plot(lon, lat, marker="*", color="#d1495b", markersize=18, markeredgecolor="k")
    ax.annotate(name, (lon, lat), xytext=(6, 6), textcoords="offset points", fontsize=11, fontweight="bold")
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.set_aspect("equal")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Earthquake locations (size = magnitude, color = depth)")
plt.colorbar(sc, ax=ax, label="Depth (km)", shrink=0.7)
plt.show()

## 3. Magnitude distribution: the Gutenberg-Richter law

Seismology predicts that each step of **+1 magnitude** has roughly **10x fewer** earthquakes:
`log10(N >= M) = a - b*M`, where the *b-value* is usually between about 0.7 and 1.2. On a log scale the cumulative count should be a straight line.

Our catalog mixes two kinds of data. Dense US regional networks report even tiny quakes. Worldwide, USGS only reliably
reports events from about **M4.5** upwards (the *magnitude of completeness*, Mc). Below Mc the curve flattens because
small quakes outside the US are simply missing, so we estimate *b* only from events at or above Mc
(Aki's maximum-likelihood method). With only a few days of data, treat the value as a rough estimate.

In [ ]:
mags = eq["magnitude"]
thresholds = np.round(np.arange(np.floor(mags.min() * 10) / 10, mags.max() + 0.1, 0.1), 1)
cum_counts = np.array([(mags >= m).sum() for m in thresholds])

MC = 4.5                                   # global completeness of the USGS feed
if (mags >= MC).sum() < 20:                # too few large events: fall back to max-curvature + 0.2
    MC = mags.round(1).mode().iloc[0] + 0.2
complete = mags[mags >= MC]
b_value = np.log10(np.e) / (complete.mean() - (MC - 0.05))    # Aki (1965), binned at 0.1
a_value = np.log10(len(complete)) + b_value * MC

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(mags, binwidth=0.2, ax=axes[0], color="#00798c")
axes[0].set_title("Magnitude histogram")
axes[0].set_xlabel("Magnitude")

axes[1].semilogy(thresholds, cum_counts, "o", markersize=4, color="#00798c", label="Observed N(>= M)")
fit_x = thresholds[thresholds >= MC]
axes[1].semilogy(fit_x, 10 ** (a_value - b_value * fit_x), "-", color="#d1495b", linewidth=2,
                 label="Gutenberg-Richter fit, b = {:.2f}".format(b_value))
axes[1].axvline(MC, color="grey", linestyle="--", label="Mc = {:.1f}".format(MC))
axes[1].set_title("Gutenberg-Richter: cumulative count (log scale)")
axes[1].set_xlabel("Magnitude M")
axes[1].set_ylabel("Number of events with magnitude >= M")
axes[1].legend()
plt.tight_layout()
plt.show()

print("Events used for the fit (M >= {:.1f}): {}".format(MC, len(complete)))
print("Estimated b-value: {:.2f}  ->  each +1 magnitude is about {:.0f}x rarer".format(b_value, 10 ** b_value))

## 4. Depth vs magnitude

Most earthquakes are **shallow** (crustal). Deep earthquakes (>300 km) only happen where an ocean plate dives under
another plate (subduction zones), such as near Japan, Tonga, Indonesia and South America.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [2, 1]})
sns.scatterplot(data=eq, x="magnitude", y="depth_km", hue="depth_class", alpha=0.6, ax=axes[0])
axes[0].invert_yaxis()
axes[0].set_title("Depth vs magnitude")
axes[0].set_xlabel("Magnitude")
axes[0].set_ylabel("Depth (km, downwards)")

eq["depth_class"].value_counts(sort=False).plot(kind="bar", ax=axes[1], color=["#edae49", "#00798c", "#30638e"])
axes[1].set_title("Events by depth class")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

## 5. Detection bias between seismic networks

Each event is reported by a seismic **network**. Regional US networks (e.g. `ak` Alaska, `ci` Southern California, `nc` Northern California)
record very small quakes because they have dense local sensors. The global `us` network mostly reports events of M2.5 or more
outside the US. **More events in a region does not always mean more seismic activity. It can mean more sensors.**

In [ ]:
net = (eq.groupby("network")["magnitude"]
         .agg(events="size", median_mag="median", max_mag="max")
         .sort_values("events", ascending=False).head(10))

fig, ax = plt.subplots(figsize=(12, 5))
top_nets = net.index.tolist()
sns.boxplot(data=eq[eq["network"].isin(top_nets)], x="network", y="magnitude", order=top_nets, color="#8ecae6", ax=ax)
for i, n in enumerate(top_nets):
    ax.text(i, eq["magnitude"].max() + 0.2, "n={}".format(net.loc[n, "events"]), ha="center", fontsize=9)
ax.set_title("Magnitude distribution by reporting network")
ax.set_xlabel("Network")
ax.set_ylabel("Magnitude")
plt.show()
net.round(2)

## 6. Activity over time

In [ ]:
span_hours = (eq["event_time"].max() - eq["event_time"].min()).total_seconds() / 3600
freq = "1h" if span_hours <= 72 else "6h"
timeline = eq.set_index("event_time").resample(freq)["magnitude"].agg(["size", "max"])

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.bar(timeline.index, timeline["size"], width=pd.Timedelta(freq) * 0.9, color="#8ecae6", label="Events")
ax1.set_ylabel("Events per {}".format(freq))
ax2 = ax1.twinx()
ax2.plot(timeline.index, timeline["max"], color="#d1495b", marker="o", markersize=3, label="Max magnitude")
ax2.set_ylabel("Max magnitude")
ax2.grid(False)
ax1.set_title("Earthquake activity over time (UTC)")
fig.legend(loc="upper right", bbox_to_anchor=(0.9, 0.88))
plt.show()

## 7. Earthquakes near Tokyo and Ho Chi Minh City

This links the earthquake stream to the cities in our weather stream. We compute the great-circle (**haversine**)
distance from every earthquake to each city. Japan sits on four plates and has frequent quakes, while southern Vietnam is far from active plate boundaries.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

for name, (lat, lon) in CITIES.items():
    eq["dist_" + name] = haversine_km(eq["latitude"], eq["longitude"], lat, lon)

radii = [250, 500, 1000, 2000]
near = pd.DataFrame({name: [(eq["dist_" + name] <= r).sum() for r in radii] for name in CITIES},
                    index=["<= {} km".format(r) for r in radii])

ax = near.plot(kind="bar", figsize=(10, 5), color=["#00798c", "#d1495b"])
ax.set_title("Number of earthquakes within a distance of each city")
ax.set_ylabel("Events")
ax.tick_params(axis="x", rotation=0)
plt.show()
near

In [ ]:
for name in CITIES:
    print("Closest 5 earthquakes to {}:".format(name))
    cols = ["place", "magnitude", "depth_km", "event_time", "dist_" + name]
    display(eq.nsmallest(5, "dist_" + name)[cols].round({"depth_km": 1, "dist_" + name: 0}))

## 8. Strongest events and most active regions

In [ ]:
strongest = eq.nlargest(10, "magnitude")[
    ["event_time", "place", "magnitude", "mag_type", "depth_km", "significance", "felt_reports", "alert_level", "tsunami"]
]
strongest

In [ ]:
regions = eq.groupby("region")["magnitude"].agg(events="size", max_mag="max", mean_mag="mean")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
regions.nlargest(12, "events")["events"].sort_values().plot(kind="barh", ax=axes[0], color="#00798c")
axes[0].set_title("Most active regions (event count)")
axes[0].set_xlabel("Events")
axes[0].set_ylabel("")

regions[regions["events"] >= 3].nlargest(12, "mean_mag")["mean_mag"].sort_values().plot(kind="barh", ax=axes[1], color="#d1495b")
axes[1].set_title("Regions with the strongest average magnitude (>= 3 events)")
axes[1].set_xlabel("Mean magnitude")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()